# 10 — Modelos de Classificação
## Logistic Regression · Naïve Bayes · SVM · KNN

**Objectivo:** Classificar o `user_persona` (5 classes) com base
no comportamento digital e hábitos de saúde.

**Modelos:**
- Logistic Regression
- Naïve Bayes (GaussianNB)
- SVM — análise de múltiplos kernels (linear, rbf, poly, sigmoid)
- KNN — análise do K óptimo

In [ ]:
# ============================================================
# SETUP
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import json, joblib, time, warnings
warnings.filterwarnings('ignore')
import sklearn
print(sklearn.__version__)
from sklearn.linear_model  import LogisticRegression
from sklearn.naive_bayes   import GaussianNB
from sklearn.svm           import SVC, LinearSVC
from sklearn.neighbors     import KNeighborsClassifier
from sklearn.metrics       import (accuracy_score, precision_score,
                                   recall_score, f1_score,
                                   confusion_matrix, classification_report,
                                   ConfusionMatrixDisplay)

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f', 'axes.facecolor': '#1a1a2e',
    'axes.labelcolor': 'white',    'xtick.color': 'white',
    'ytick.color': 'white',        'text.color': 'white',
    'axes.titlecolor': 'white',    'grid.color': '#2a2a4a',
    'axes.spines.top': False,      'axes.spines.right': False,
})
COLORS = ['#833ab4','#fd1d1d','#fcb045','#405de6','#5851db','#e1306c','#f77737']

print('✅ Setup completo!')

✅ Setup completo!


In [2]:
# ============================================================
# CARREGAMENTO DOS DADOS DO NOTEBOOK 08
# ============================================================
X_train     = np.load('../models/X_train.npy')
X_test      = np.load('../models/X_test.npy')
X_train_std = np.load('../models/X_train_std.npy')
X_test_std  = np.load('../models/X_test_std.npy')
y_train     = np.load('../models/y_train_clf.npy')
y_test      = np.load('../models/y_test_clf.npy')

le = joblib.load('../models/label_encoder_persona.pkl')
with open('../models/feature_names.json') as f:
    FEATURES = json.load(f)
with open('../models/ml_config.json') as f:
    config = json.load(f)

CLASSES     = le.classes_
SAMPLES     = config['SAMPLES']

print(f'✅ Dados carregados:')
print(f'   X_train : {X_train.shape}')
print(f'   X_test  : {X_test.shape}')
print(f'   Classes : {CLASSES.tolist()}')
print(f'\n   Distribuição y_test:')
unique, counts = np.unique(y_test, return_counts=True)
for u, c in zip(unique, counts):
    print(f'     {le.inverse_transform([u])[0]:<25}: {c:>5} ({c/len(y_test)*100:.1f}%)')

✅ Dados carregados:
   X_train : (40000, 19)
   X_test  : (10000, 19)
   Classes : ['Doom-Scroller', 'Influencer/Creator', 'Silent Browser', 'Social Poster', 'Utilizador Casual']

   Distribuição y_test:
     Doom-Scroller            :    23 (0.2%)
     Influencer/Creator       :  4829 (48.3%)
     Silent Browser           :   325 (3.2%)
     Social Poster            :  1241 (12.4%)
     Utilizador Casual        :  3582 (35.8%)


In [3]:
# ============================================================
# FUNÇÃO DE AVALIAÇÃO
# ============================================================
def avaliar_classificador(nome, y_true, y_pred, tempo_treino, classes):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f'\n{"="*60}')
    print(f'  📊 {nome}')
    print(f'{"="*60}')
    print(f'  Accuracy          : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  Precision (wtd)   : {prec:.4f}')
    print(f'  Recall    (wtd)   : {rec:.4f}')
    print(f'  F1-Score  (wtd)   : {f1:.4f}')
    print(f'  Tempo treino      : {tempo_treino:.4f}s')
    print(f'\n  Classification Report:')
    print(classification_report(y_true, y_pred,
                                target_names=classes, zero_division=0))
    return {
        'Modelo': nome, 'Accuracy': round(acc,4),
        'Precision': round(prec,4), 'Recall': round(rec,4),
        'F1': round(f1,4), 'Tempo(s)': round(tempo_treino,4)
    }

resultados = []

In [4]:
# ============================================================
# FUNÇÃO MATRIZ DE CONFUSÃO
# ============================================================
def plot_confusion_matrix(ax, y_true, y_pred, classes, titulo, color):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]

    im = ax.imshow(cm_norm, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(
        [c.replace('/','\n').replace(' ','\n') for c in classes],
        fontsize=7, rotation=30, ha='right'
    )
    ax.set_yticklabels(
        [c.replace('/','\n').replace(' ','\n') for c in classes],
        fontsize=7
    )
    ax.set_xlabel('Previsto', fontsize=9)
    ax.set_ylabel('Real',     fontsize=9)
    ax.set_title(titulo, fontsize=10)

    for i in range(len(classes)):
        for j in range(len(classes)):
            val  = cm_norm[i, j]
            text = f'{val:.2f}\n({cm[i,j]})'
            ax.text(j, i, text, ha='center', va='center',
                    color='white' if val > 0.5 else 'black',
                    fontsize=7)
    return im

In [5]:
# ============================================================
# MODELO 1 — LOGISTIC REGRESSION
# ============================================================
print('🔧 A treinar Logistic Regression...')

t0  = time.time()
lr  = LogisticRegression(max_iter=1000, random_state=42,
                         multi_class='auto', solver='lbfgs')
lr.fit(X_train_std, y_train)
t1  = time.time()

y_pred_lr = lr.predict(X_test_std)
res_lr    = avaliar_classificador('Logistic Regression',
                                   y_test, y_pred_lr, t1-t0, CLASSES)
resultados.append(res_lr)

# Probabilidades — confiança do modelo
proba_lr = lr.predict_proba(X_test_std)
print(f'\n📊 Confiança média por classe:')
for i, cls in enumerate(CLASSES):
    print(f'   {cls:<25}: {proba_lr[:,i].mean():.3f}')

joblib.dump(lr, '../models/logistic_regression.pkl')
print('\n💾 Modelo guardado: models/logistic_regression.pkl')

🔧 A treinar Logistic Regression...


TypeError: LogisticRegression.__init__() got an unexpected keyword argument 'multi_class'

In [ ]:
# ============================================================
# MODELO 2 — NAÏVE BAYES (GaussianNB)
# ============================================================
print('🔧 A treinar Naïve Bayes...')

t0  = time.time()
nb  = GaussianNB()
nb.fit(X_train, y_train)          # NB não precisa de normalização
t1  = time.time()

y_pred_nb = nb.predict(X_test)
res_nb    = avaliar_classificador('Naïve Bayes (Gaussian)',
                                   y_test, y_pred_nb, t1-t0, CLASSES)
resultados.append(res_nb)

print(f'\n📊 Probabilidades a priori (π):')
for cls, prior in zip(CLASSES, nb.class_prior_):
    print(f'   {cls:<25}: {prior:.4f}')

joblib.dump(nb, '../models/naive_bayes.pkl')
print('\n💾 Modelo guardado: models/naive_bayes.pkl')

In [ ]:
# ============================================================
# MODELO 3 — SVM: ANÁLISE DE MÚLTIPLOS KERNELS
# ⚠️  Usa amostra reduzida — SVM é O(n² a n³)
# ============================================================
print(f'⚠️  SVM usa amostra de {SAMPLES["SVM"]:,} linhas (complexidade O(n²))')
print('🔧 A treinar SVM com múltiplos kernels...\n')

# Cria amostra para SVM
idx_svm     = np.random.RandomState(42).choice(
    len(X_train_std), size=min(SAMPLES['SVM'], len(X_train_std)), replace=False
)
X_svm_train = X_train_std[idx_svm]
y_svm_train = y_train[idx_svm]

kernels      = ['linear', 'rbf', 'poly', 'sigmoid']
svm_results  = []
svm_models   = {}

for kernel in kernels:
    print(f'   Kernel: {kernel:<10}', end=' ')
    t0  = time.time()
    svm = SVC(kernel=kernel, random_state=42, probability=True,
              C=1.0, gamma='scale')
    svm.fit(X_svm_train, y_svm_train)
    t1  = time.time()

    y_pred  = svm.predict(X_test_std)
    acc     = accuracy_score(y_test, y_pred)
    f1      = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    tempo   = t1 - t0
    svm_results.append({'Kernel': kernel, 'Accuracy': round(acc,4),
                        'F1': round(f1,4), 'Tempo(s)': round(tempo,4)})
    svm_models[kernel] = (svm, y_pred)
    print(f'Acc={acc:.4f}  F1={f1:.4f}  Tempo={tempo:.2f}s')

svm_df = pd.DataFrame(svm_results)
print(f'\n📊 COMPARAÇÃO DE KERNELS SVM:')
print(svm_df.to_string(index=False))

# Melhor kernel
best_kernel = svm_df.loc[svm_df['F1'].idxmax(), 'Kernel']
print(f'\n✅ Melhor kernel: {best_kernel}')

# Avalia o melhor
best_svm, y_pred_svm = svm_models[best_kernel]
res_svm = avaliar_classificador(f'SVM (kernel={best_kernel})',
                                 y_test, y_pred_svm,
                                 svm_df[svm_df['Kernel']==best_kernel]['Tempo(s)'].values[0],
                                 CLASSES)
resultados.append(res_svm)

joblib.dump(best_svm, '../models/svm_classifier.pkl')
print(f'\n💾 Modelo guardado: models/svm_classifier.pkl')

In [ ]:
# ============================================================
# GRÁFICO — COMPARAÇÃO DE KERNELS SVM
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('🔵 SVM — Comparação de Kernels', fontsize=14, fontweight='bold')

metricas_svm = ['Accuracy', 'F1', 'Tempo(s)']
for ax, met in zip(axes, metricas_svm):
    bars = ax.bar(svm_df['Kernel'], svm_df[met],
                  color=COLORS[:4], alpha=0.85)
    best_val = svm_df[met].max() if met != 'Tempo(s)' else svm_df[met].min()
    best_k   = (svm_df.loc[svm_df[met].idxmax(), 'Kernel']
                if met != 'Tempo(s)' else
                svm_df.loc[svm_df[met].idxmin(), 'Kernel'])
    for bar, val, kern in zip(bars, svm_df[met], svm_df['Kernel']):
        clr = 'yellow' if kern == best_k else 'white'
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height() + svm_df[met].max()*0.01,
                f'{val:.4f}', ha='center', color=clr, fontsize=10,
                fontweight='bold' if kern == best_k else 'normal')
    ax.set_title(met, fontsize=12)
    ax.set_xlabel('Kernel')
    ax.set_ylabel(met)
    ax.tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.savefig('../data/fig_svm_kernels.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('💾 Guardado: fig_svm_kernels.png')

In [ ]:
# ============================================================
# MODELO 4 — KNN: ANÁLISE DO K ÓPTIMO
# ⚠️  Usa amostra reduzida
# ============================================================
print(f'⚠️  KNN usa amostra de {SAMPLES["KNN"]:,} linhas')
print('🔧 A analisar KNN com K de 1 a 30...\n')

idx_knn     = np.random.RandomState(42).choice(
    len(X_train_std), size=min(SAMPLES['KNN'], len(X_train_std)), replace=False
)
X_knn_train = X_train_std[idx_knn]
y_knn_train = y_train[idx_knn]

K_values  = list(range(1, 31))
knn_accs  = []
knn_f1s   = []
knn_times = []

for k in K_values:
    t0   = time.time()
    knn  = KNeighborsClassifier(n_neighbors=k, metric='euclidean', n_jobs=-1)
    knn.fit(X_knn_train, y_knn_train)
    t1   = time.time()
    y_pred_k = knn.predict(X_test_std)
    knn_accs.append(accuracy_score(y_test, y_pred_k))
    knn_f1s.append(f1_score(y_test, y_pred_k, average='weighted', zero_division=0))
    knn_times.append(t1 - t0)
    if k % 5 == 0:
        print(f'   K={k:>2}: Acc={knn_accs[-1]:.4f}  F1={knn_f1s[-1]:.4f}')

best_k   = K_values[np.argmax(knn_f1s)]
print(f'\n✅ K óptimo: {best_k}  (F1={max(knn_f1s):.4f})')

# Treina com K óptimo
t0      = time.time()
knn_best = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean', n_jobs=-1)
knn_best.fit(X_knn_train, y_knn_train)
t1      = time.time()
y_pred_knn = knn_best.predict(X_test_std)
res_knn    = avaliar_classificador(f'KNN (K={best_k})',
                                    y_test, y_pred_knn, t1-t0, CLASSES)
resultados.append(res_knn)

joblib.dump(knn_best, '../models/knn_classifier.pkl')
print(f'\n💾 Modelo guardado: models/knn_classifier.pkl')

In [ ]:
# ============================================================
# GRÁFICO — KNN: CURVA DO K ÓPTIMO
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'🔍 KNN — Análise do K Óptimo (K={best_k})',
             fontsize=14, fontweight='bold')

# Accuracy e F1 vs K
axes[0].plot(K_values, knn_accs, color=COLORS[0], lw=2.5,
             marker='o', markersize=5, label='Accuracy')
axes[0].plot(K_values, knn_f1s,  color=COLORS[2], lw=2.5,
             marker='s', markersize=5, label='F1-Score')
axes[0].axvline(best_k, color='white', ls='--', lw=2,
                label=f'K óptimo = {best_k}')
axes[0].set_title('Accuracy e F1 vs K', fontsize=12)
axes[0].set_xlabel('Número de Vizinhos (K)')
axes[0].set_ylabel('Score')
axes[0].legend(fontsize=9)
axes[0].set_xticks(K_values[::2])

# Overfitting: K pequeno = overfit, K grande = underfit
ax_er = axes[0].twinx()
ax_er.plot(K_values, [1-a for a in knn_accs], color=COLORS[1],
           lw=1.5, ls=':', alpha=0.6, label='Erro')
ax_er.set_ylabel('Taxa de Erro', color=COLORS[1])
ax_er.tick_params(axis='y', colors=COLORS[1])

# Tempo vs K
axes[1].plot(K_values, knn_times, color=COLORS[4], lw=2.5,
             marker='D', markersize=5)
axes[1].axvline(best_k, color='white', ls='--', lw=2,
                label=f'K óptimo = {best_k}')
axes[1].set_title('Tempo de Treino vs K', fontsize=12)
axes[1].set_xlabel('Número de Vizinhos (K)')
axes[1].set_ylabel('Tempo (s)')
axes[1].legend(fontsize=9)
axes[1].set_xticks(K_values[::2])

plt.tight_layout()
plt.savefig('../data/fig_knn_optimal_k.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('💾 Guardado: fig_knn_optimal_k.png')

In [ ]:
# ============================================================
# MATRIZES DE CONFUSÃO — 4 MODELOS
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('🗂️  Matrizes de Confusão (Normalizada)',
             fontsize=15, fontweight='bold')
axes = axes.flatten()

preds = [
    ('Logistic Regression', y_pred_lr, COLORS[0]),
    ('Naïve Bayes',          y_pred_nb, COLORS[1]),
    (f'SVM ({best_kernel})', y_pred_svm,COLORS[3]),
    (f'KNN (K={best_k})',    y_pred_knn,COLORS[4]),
]

for ax, (nome, y_pred, cor) in zip(axes, preds):
    plot_confusion_matrix(ax, y_test, y_pred, CLASSES, nome, cor)

plt.tight_layout()
plt.savefig('../data/fig_confusion_matrices.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('💾 Guardado: fig_confusion_matrices.png')

In [ ]:
# ============================================================
# COMPARAÇÃO FINAL DOS 4 MODELOS
# ============================================================
results_df = pd.DataFrame(resultados)

print('📊 COMPARAÇÃO FINAL — MODELOS DE CLASSIFICAÇÃO')
print('='*70)
print(results_df.to_string(index=False))

best_model_name = results_df.loc[results_df['F1'].idxmax(), 'Modelo']
best_acc        = results_df.loc[results_df['Accuracy'].idxmax(), 'Modelo']
print(f'\n🏆 Melhor F1        : {results_df.loc[results_df["F1"].idxmax(), "Modelo"]}  ({results_df["F1"].max():.4f})')
print(f'🏆 Melhor Accuracy  : {best_acc}  ({results_df["Accuracy"].max():.4f})')
print(f'⚡ Mais rápido      : {results_df.loc[results_df["Tempo(s)"].idxmin(), "Modelo"]}')

# Gráfico radar de comparação
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
fig.suptitle('📊 Comparação Final — Modelos de Classificação',
             fontsize=14, fontweight='bold')

metricas = ['Accuracy', 'Precision', 'Recall', 'F1']
for ax, met in zip(axes, metricas):
    sorted_df = results_df.sort_values(met, ascending=True)
    colors_bar = [COLORS[3] if m == best_model_name else COLORS[0]
                  for m in sorted_df['Modelo']]
    bars = ax.barh(sorted_df['Modelo'], sorted_df[met],
                   color=colors_bar, alpha=0.85)
    for bar, val in zip(bars, sorted_df[met]):
        ax.text(val + 0.002, bar.get_y()+bar.get_height()/2,
                f'{val:.4f}', va='center', color='white', fontsize=9)
    ax.set_title(met, fontsize=12)
    ax.set_xlabel(met)
    ax.set_xlim(0, min(1.0, sorted_df[met].max()*1.12))
    ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig('../data/fig_classification_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('💾 Guardado: fig_classification_comparison.png')

In [ ]:
# ============================================================
# GUARDAR RESULTADOS
# ============================================================
results_df.to_csv('../data/classification_results.csv', index=False)

# Melhor modelo de classificação
best_row = results_df.loc[results_df['F1'].idxmax()]
if 'Logistic' in best_row['Modelo']:
    best_clf, best_preds_clf = lr, y_pred_lr
elif 'Naïve' in best_row['Modelo']:
    best_clf, best_preds_clf = nb, y_pred_nb
elif 'SVM' in best_row['Modelo']:
    best_clf, best_preds_clf = best_svm, y_pred_svm
else:
    best_clf, best_preds_clf = knn_best, y_pred_knn

joblib.dump(best_clf, '../models/best_clf_notebook10.pkl')
np.save('../models/y_pred_classification.npy', best_preds_clf)

print('💾 Ficheiros guardados:')
print('   ✅ data/classification_results.csv')
print('   ✅ models/best_clf_notebook10.pkl')
print('   ✅ models/y_pred_classification.npy')
print('   ✅ models/logistic_regression.pkl')
print('   ✅ models/naive_bayes.pkl')
print('   ✅ models/svm_classifier.pkl')
print('   ✅ models/knn_classifier.pkl')

print(f'\n✅ Notebook 10 completo!')
print(f'   Próximo: 11_tree_ensemble_models.ipynb')

In [ ]:
# ============================================================
# INTERPRETAÇÃO DOS RESULTADOS
# ============================================================
print('='*65)
print('   📝 INTERPRETAÇÃO')
print('='*65)
print(f"""
LOGISTIC REGRESSION:
  • Modelo linear para classificação — fronteiras de decisão lineares
  • Produz probabilidades calibradas por classe
  • Funciona bem quando as classes são linearmente separáveis
  • Rápido e interpretável — coeficientes indicam importância das features

NAÏVE BAYES (Gaussian):
  • Assume independência condicional entre as features (pressuposto naïve)
  • Extremamente rápido — ideal para baseline
  • Funciona surpreendentemente bem em muitos casos reais
  • Menos preciso quando as features têm correlações fortes entre si
    (o que acontece no nosso dataset — digital_addiction correlaciona
    com muitas outras features)

SVM ({best_kernel} kernel):
  • Melhor kernel: {best_kernel}
  • O kernel RBF é geralmente o melhor para dados não lineares
  • O kernel linear é mais rápido mas menos preciso em dados complexos
  • C controla o trade-off entre margem e classificações erradas
  • Nota: usámos {SAMPLES['SVM']:,} amostras — com o dataset completo
    os resultados poderiam ser ligeiramente diferentes

KNN (K={best_k}):
  • K óptimo = {best_k} — balança bias-variance
  • K pequeno → overfit (memoriza os dados de treino)
  • K grande → underfit (decisão demasiado suave)
  • Sensível à escala das features — sempre usar StandardScaler
  • Computacionalmente pesado na fase de predição (O(n) por sample)

CONCLUSÃO:
  • Melhor modelo por F1: {results_df.loc[results_df['F1'].idxmax(), 'Modelo']}
  • Para uso em produção (dashboard): {results_df.loc[results_df['F1'].idxmax(), 'Modelo']}
    pela combinação de velocidade e precisão
""")